# ViT + LSTM Temporal Modeling for Gesture Recognition

This notebook extends the existing ViT gesture-classification pipeline with a temporal head. It keeps the ViT embedding interface stable and makes the temporal head swappable (LSTM now, Mamba later).

In [ ]:
# 1) Set Up Environment and Imports
# Keep this minimal; rely on your conda env (environment.yml). Fallback to pip if needed.
import sys, subprocess, os

REQUIRED = [
    ("torch", None),
    ("timm", None),
    ("torchvision", None),
    ("sklearn", None),
    ("matplotlib", None),
]

for pkg, extra in REQUIRED:
    try:
        __import__(pkg)
    except Exception:
        if os.environ.get("ALLOW_PIP_FALLBACK", "1") == "1":
            print(f"Installing missing package: {pkg}")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        else:
            raise

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
import timm
import numpy as np
import random
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("CUDA:", torch.cuda.get_device_name(0))
    print("Compute capability available.")


In [ ]:
# 2) Load Existing ViT and Lock Feature Interface
from typing import Optional, Tuple

class ViTBackbone(nn.Module):
    """Wrap timm ViT and expose a D-dim embedding (e.g., 768 for vit_base_patch16_224)."""
    def __init__(self, name: str = 'vit_base_patch16_224', pretrained: bool = True):
        super().__init__()
        self.model = timm.create_model(name, pretrained=pretrained)
        # Remove classification head to get feature embeddings
        if hasattr(self.model, 'head'):
            self.model.head = nn.Identity()
        elif hasattr(self.model, 'fc'):
            self.model.fc = nn.Identity()
        self.model.eval()

        # Infer embedding dim (e.g., 768)
        self.embedding_dim = getattr(self.model, 'num_features', 768)

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 3, H, W) -> (B, D)
        return self.model(x)


def get_vit_backbone(name: str = 'vit_base_patch16_224', pretrained: bool = True, checkpoint_path: Optional[str] = None) -> Tuple[ViTBackbone, int]:
    vit = ViTBackbone(name=name, pretrained=pretrained)
    if checkpoint_path and os.path.exists(checkpoint_path):
        state = torch.load(checkpoint_path, map_location='cpu')
        missing, unexpected = vit.load_state_dict(state, strict=False)
        print("Loaded ViT checkpoint:", checkpoint_path)
        print("Missing keys:", missing)
        print("Unexpected keys:", unexpected)
    D = vit.embedding_dim
    print(f"ViT embedding dim D={D}")
    return vit, D

vit, D = get_vit_backbone(pretrained=True)
for p in vit.parameters():
    p.requires_grad = False
print("ViT frozen.")


In [ ]:
# 3) Define Preprocessing and Normalization Aligned with ViT
vit_img_size = 224
vit_mean = (0.485, 0.456, 0.406)
vit_std = (0.229, 0.224, 0.225)

preprocess = transforms.Compose([
    transforms.Resize((vit_img_size, vit_img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=vit_mean, std=vit_std),
])


def preprocess_frame(pil_image):
    return preprocess(pil_image)


In [ ]:
# 4) ViTFeatureExtractor: [B,T,C,H,W] -> [B,T,D]
class ViTFeatureExtractor(nn.Module):
    def __init__(self, vit_backbone: ViTBackbone):
        super().__init__()
        self.vit = vit_backbone
        self.D = vit_backbone.embedding_dim

    @torch.no_grad()
    def forward(self, frames: torch.Tensor, lengths: Optional[torch.Tensor] = None) -> torch.Tensor:
        # frames: (B, T, C, H, W) -> embeddings: (B, T, D)
        B, T = frames.shape[:2]
        flat = frames.view(B * T, *frames.shape[2:])
        feats = self.vit(flat)  # (B*T, D)
        feats = feats.view(B, T, self.D)
        return feats

vit_extractor = ViTFeatureExtractor(vit).to(device)
vit_extractor.eval()


In [ ]:
# 5) TemporalHead Interface (Swappable)
class TemporalHead(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.input_dim = input_dim
        self.num_classes = num_classes

    def forward(self, x: torch.Tensor, lengths: Optional[torch.Tensor] = None) -> torch.Tensor:
        # x: (B, T, D) -> logits: (B, K)
        raise NotImplementedError

HEAD_REGISTRY = {}

def register_head(name):
    def deco(cls):
        HEAD_REGISTRY[name] = cls
        return cls
    return deco


In [ ]:
# 6) LSTMHead Implementation
@register_head('lstm')
class LSTMHead(TemporalHead):
    def __init__(self, input_dim: int, num_classes: int, hidden_dim: int = 384, num_layers: int = 2, bidirectional: bool = True, dropout: float = 0.2):
        super().__init__(input_dim=input_dim, num_classes=num_classes)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        out_dim = hidden_dim * (2 if bidirectional else 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(out_dim, num_classes)

    def forward(self, x: torch.Tensor, lengths: Optional[torch.Tensor] = None) -> torch.Tensor:
        # x: (B, T, D)
        if lengths is not None:
            # pack sequences for efficiency
            packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            h, _ = self.lstm(packed)
            h, _ = nn.utils.rnn.pad_packed_sequence(h, batch_first=True)
            # gather last valid timestep per sequence
            idx = (lengths - 1).clamp(min=0).view(-1, 1, 1).expand(-1, 1, h.size(2))
            last = h.gather(dim=1, index=idx).squeeze(1)
        else:
            h, _ = self.lstm(x)
            last = h[:, -1, :]
        logits = self.classifier(self.dropout(last))
        return logits

# quick unit test
with torch.no_grad():
    test_x = torch.randn(4, 10, D)
    test_len = torch.tensor([10, 8, 6, 10])
    head = LSTMHead(input_dim=D, num_classes=5)
    out = head(test_x, test_len)
    print("LSTMHead output shape:", tuple(out.shape))


In [ ]:
# 7) Model Assembly: ViTBackbone + TemporalHead
class ViTTemporalModel(nn.Module):
    def __init__(self, vit_extractor: ViTFeatureExtractor, head_name: str, num_classes: int, **head_kwargs):
        super().__init__()
        self.vit_extractor = vit_extractor
        self.D = vit_extractor.D
        head_cls = HEAD_REGISTRY[head_name]
        self.temporal_head = head_cls(input_dim=self.D, num_classes=num_classes, **head_kwargs)

    def freeze_vit(self):
        for p in self.vit_extractor.parameters():
            p.requires_grad = False

    def unfreeze_vit(self):
        for p in self.vit_extractor.parameters():
            p.requires_grad = True

    def forward(self, frames: torch.Tensor, lengths: Optional[torch.Tensor] = None) -> torch.Tensor:
        with torch.no_grad():
            feats = self.vit_extractor(frames, lengths)  # (B,T,D)
        logits = self.temporal_head(feats, lengths)
        return logits

model = ViTTemporalModel(vit_extractor, head_name='lstm', num_classes=5, hidden_dim=384, num_layers=2, bidirectional=True, dropout=0.2).to(device)
print(model)


In [ ]:
# 8) Dataset: Synthetic Sequence Dataset (runnable fallback)
from PIL import Image

class SyntheticGestureDataset(Dataset):
    """Produces synthetic sequences to test static vs dynamic training.
    - K classes
    - Static: constant class across T
    - Dynamic: single transition i->j halfway
    Frames are random noise passed through ViT preprocess for shape/normalization only.
    """
    def __init__(self, num_samples: int, T: int, K: int, dynamic_ratio: float = 0.5):
        self.num_samples = num_samples
        self.T = T
        self.K = K
        self.dynamic_ratio = dynamic_ratio

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        is_dynamic = (random.random() < self.dynamic_ratio)
        if is_dynamic and self.K >= 2:
            c1 = random.randrange(self.K)
            c2 = (c1 + random.randrange(1, self.K)) % self.K
            label = c2  # sequence label by final class (simplification)
        else:
            c1 = random.randrange(self.K)
            c2 = c1
            label = c1

        frames = []
        for t in range(self.T):
            # Random noise image; in real dataset, load ROI frame then preprocess
            arr = np.uint8(np.random.rand(vit_img_size, vit_img_size, 3) * 255)
            img = Image.fromarray(arr)
            frame = preprocess(img)
            frames.append(frame)
        frames = torch.stack(frames, dim=0)  # (T,C,H,W)
        length = torch.tensor(self.T, dtype=torch.long)
        return frames, label, length


def collate_sequences(batch):
    frames, labels, lengths = zip(*batch)
    T = max(l.item() for l in lengths)
    B = len(batch)
    C, H, W = frames[0].shape[1:]
    out = torch.zeros(B, T, C, H, W)
    for i, (f, l) in enumerate(zip(frames, lengths)):
        out[i, : l, ...] = f[: l]
    labels = torch.tensor(labels, dtype=torch.long)
    lengths = torch.stack(lengths)
    mask = torch.arange(T).unsqueeze(0) < lengths.unsqueeze(1)
    return out, labels, lengths, mask


In [ ]:
# 9) Config Boundaries
from dataclasses import dataclass

@dataclass
class Config:
    vit_name: str = 'vit_base_patch16_224'
    vit_img_size: int = 224
    vit_frozen: bool = True
    embedding_dim: int = D
    head_type: str = 'lstm'
    head_hidden_dim: int = 384
    head_layers: int = 2
    head_bidirectional: bool = True
    head_dropout: float = 0.2
    num_classes: int = 5
    batch_size: int = 8
    epochs: int = 2
    lr: float = 1e-3
    weight_decay: float = 1e-4
    mixed_precision: bool = False
    train_T: int = 16
    val_T: int = 16
    num_workers: int = 0

cfg = Config()
print(cfg)


In [ ]:
# 10) Build Dataloaders
train_ds = SyntheticGestureDataset(num_samples=200, T=cfg.train_T, K=cfg.num_classes, dynamic_ratio=0.5)
val_ds = SyntheticGestureDataset(num_samples=60, T=cfg.val_T, K=cfg.num_classes, dynamic_ratio=0.5)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, collate_fn=collate_sequences)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, collate_fn=collate_sequences)

len(train_loader), len(val_loader)


In [ ]:
# 11) Training / Validation Loops
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
criterion = nn.CrossEntropyLoss()

scaler = torch.cuda.amp.GradScaler(enabled=cfg.mixed_precision and device.type=="cuda")

def train_one_epoch(epoch:int):
    model.train()
    running_loss = 0.0
    total, correct = 0, 0
    for frames, labels, lengths, mask in train_loader:
        frames, labels, lengths = frames.to(device), labels.to(device), lengths.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=cfg.mixed_precision and device.type=="cuda"):
            logits = model(frames, lengths)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * frames.size(0)
        preds = logits.argmax(dim=1)
        total += labels.numel()
        correct += (preds == labels).sum().item()
    print(f"Epoch {epoch} train loss={running_loss/len(train_ds):.4f} acc={correct/total:.3f}")

def evaluate():
    model.eval()
    total, correct = 0, 0
    all_labels, all_preds = [], []
    with torch.no_grad():
        for frames, labels, lengths, mask in val_loader:
            frames, labels, lengths = frames.to(device), labels.to(device), lengths.to(device)
            logits = model(frames, lengths)
            preds = logits.argmax(dim=1)
            total += labels.numel()
            correct += (preds == labels).sum().item()
            all_labels.extend(labels.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())
    acc = correct/total
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(cfg.num_classes)))
    print("Val acc=%.3f" % acc)
    print("Confusion matrix:\n", cm)
    return acc, cm

for epoch in range(1, cfg.epochs+1):
    train_one_epoch(epoch)
    evaluate()


In [ ]:
# 12) Static vs Dynamic Training Runs Demo
# For a quick sanity check, run a single eval on T=1 vs T=16 by reusing loaders or changing cfg.
# In real use, you will prepare datasets with real frame sequences per gesture.

# Already trained briefly above on mixed static/dynamic synthetic data.
# Here we just demonstrate inference on a new synthetic batch.

val_ds_short = SyntheticGestureDataset(num_samples=16, T=1, K=cfg.num_classes, dynamic_ratio=0.0)
val_loader_short = DataLoader(val_ds_short, batch_size=8, shuffle=False, collate_fn=collate_sequences)

model.eval()
with torch.no_grad():
    for frames, labels, lengths, mask in val_loader_short:
        frames, labels, lengths = frames.to(device), labels.to(device), lengths.to(device)
        logits = model(frames, lengths)
        preds = logits.argmax(dim=1)
        print("Static (T=1) batch acc:", (preds==labels).float().mean().item())
        break

val_ds_long = SyntheticGestureDataset(num_samples=16, T=16, K=cfg.num_classes, dynamic_ratio=0.5)
val_loader_long = DataLoader(val_ds_long, batch_size=8, shuffle=False, collate_fn=collate_sequences)

with torch.no_grad():
    for frames, labels, lengths, mask in val_loader_long:
        frames, labels, lengths = frames.to(device), labels.to(device), lengths.to(device)
        logits = model(frames, lengths)
        preds = logits.argmax(dim=1)
        print("Dynamic (T=16) batch acc:", (preds==labels).float().mean().item())
        break


In [ ]:
# 13) Temporal Semantics: Inspect Per-frame Evidence
# Extract embeddings and see how logits evolve across T for a sample

model.eval()
with torch.no_grad():
    sample_ds = SyntheticGestureDataset(num_samples=1, T=16, K=cfg.num_classes, dynamic_ratio=0.5)
    frames, label, lengths = sample_ds[0]
    frames = frames.unsqueeze(0).to(device)  # (1,T,C,H,W)
    lengths = lengths.unsqueeze(0).to(device)
    feats = vit_extractor(frames, lengths)  # (1,T,D)
    logits = model.temporal_head(feats, lengths)  # (1,K)
    print("Sequence label:", int(label), "Logits shape:", tuple(logits.shape))


In [ ]:
# 14) Future Extensibility: MambaHead Stub (drop-in)
class MambaHead(TemporalHead):
    def __init__(self, input_dim: int, num_classes: int, **kwargs):
        super().__init__(input_dim=input_dim, num_classes=num_classes)
        # Placeholder: keep identical interface; implement with mamba-ssm later
        self.proj = nn.Linear(input_dim, num_classes)

    def forward(self, x: torch.Tensor, lengths: Optional[torch.Tensor] = None) -> torch.Tensor:
        # Simple pooled baseline to illustrate drop-in compatibility
        pooled = x.mean(dim=1)
        return self.proj(pooled)

register_head('mamba')(MambaHead)

# Demonstrate swap without changing ViT or dataset
mamba_model = ViTTemporalModel(vit_extractor, head_name='mamba', num_classes=cfg.num_classes).to(device)
print("Swapped to MambaHead-compatible model (stub).")


In [ ]:
# 15) Save/Load Checkpoint and Export Interface Docs
import json

ckpt_path = "./checkpoints/vit_lstm_synthetic.pt"
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

def save_checkpoint(model: ViTTemporalModel, cfg: Config, path: str):
    torch.save({
        'state_dict': model.state_dict(),
        'cfg': cfg.__dict__,
        'D': model.D,
    }, path)
    print("Saved:", path)


def load_checkpoint(path: str) -> Tuple[ViTTemporalModel, Config]:
    blob = torch.load(path, map_location='cpu')
    cfg = Config(**blob['cfg'])
    # Recreate ViT and head
    vit2, _D = get_vit_backbone(pretrained=True)
    vit_ex2 = ViTFeatureExtractor(vit2)
    model2 = ViTTemporalModel(vit_ex2, head_name=cfg.head_type, num_classes=cfg.num_classes, hidden_dim=cfg.head_hidden_dim, num_layers=cfg.head_layers, bidirectional=cfg.head_bidirectional, dropout=cfg.head_dropout)
    model2.load_state_dict(blob['state_dict'], strict=False)
    return model2, cfg

# Save once after short training
save_checkpoint(model, cfg, ckpt_path)

# Export concise interface doc
doc = {
    'vit_embedding_interface': {
        'input': '[B,T,3,H,W]',
        'output': '[B,T,D]',
        'D': int(D),
        'normalization': {'mean': vit_mean, 'std': vit_std},
    },
    'temporal_head': {
        'input': '[B,T,D]',
        'output': '[B,K]',
        'swap_examples': ['lstm', 'mamba'],
    },
    'static_vs_dynamic': 'ViT provides per-frame evidence; the temporal head aggregates over time to classify sequences and can optionally emit per-frame logits.',
}

doc_path = "./docs/vit_lstm_interface.json"
os.makedirs(os.path.dirname(doc_path), exist_ok=True)
with open(doc_path, 'w') as f:
    json.dump(doc, f, indent=2)
print("Wrote interface doc:", doc_path)
